# Lab 03.4 — Context-based Control (P8)

## Overview

P0–P7 são **identity-based**: the decision depends ONLY on the principal and the action.

P8 is **context-based** — depende tambism dos **parâmetros da chamada**:

```cedar
forbid(...)  # create_work_order
when {
  context.input.priority == "high" &&
  !(principal.getTag("cognito:groups") like "*managers*")
};
```

Ou seja: Ana pode criar work orders normais, mas se ela tentar com
`priority=high`, is negado.

## Prerequisites

- ✅ Lab 03.2 (engine atrelado) e Lab 03.3 (já has tokens)

## Setup

In [ ]:
%pip install --quiet nest_asyncio
import sys
import json
sys.path.insert(0, "..")
from shared.utils.config import load_config, get_region
from utils import test_authorize_action

import importlib.util
spec = importlib.util.spec_from_file_location("identity_utils", "../01-Identity-Foundation/utils.py")
identity_utils = importlib.util.module_from_spec(spec)
spec.loader.exec_module(identity_utils)

cfg = load_config()
region = get_region()
mcp_url = cfg["GATEWAY_URL"]

ana_token = identity_utils.get_bearer_token(
    pool_id=cfg["COGNITO_USER_POOL_ID"], client_id=cfg["COGNITO_CLIENT_ID"],
    username="ana.operadora@workshop.local", password="Workshop@2025!", region=region,
)["access_token"]
carlos_token = identity_utils.get_bearer_token(
    pool_id=cfg["COGNITO_USER_POOL_ID"], client_id=cfg["COGNITO_CLIENT_ID"],
    username="carlos.gestor@workshop.local", password="Workshop@2025!", region=region,
)["access_token"]

## Cenário A: Ana cria work order **medium** — PERMIT (P0)

In [ ]:
r = test_authorize_action(
    mcp_url=mcp_url,
    bearer_token=ana_token,
    action="maintenanceapi___create_work_order",
    arguments={"asset_id": "SE-LESTE-03", "description": "Inspeção rotineira", "priority": "medium"},
)
print(json.dumps(r, indent=2)[:500])
assert r["decision"] == "ALLOW"
print("\n✓ P0 permite Ana com priority=medium")

## Cenário B: Ana cria work order **high** — DENY (P8 dispara)

In [ ]:
r = test_authorize_action(
    mcp_url=mcp_url,
    bearer_token=ana_token,
    action="maintenanceapi___create_work_order",
    arguments={"asset_id": "SE-LESTE-03", "description": "Substituição urgente", "priority": "high"},
)
print(json.dumps(r, indent=2)[:500])
assert r["decision"] == "DENY"
print("\n✓ P8 bloqueia Ana com priority=high")

## Cenário C: Carlos cria high — PERMIT (P8 não dispara)

In [ ]:
r = test_authorize_action(
    mcp_url=mcp_url,
    bearer_token=carlos_token,
    action="maintenanceapi___create_work_order",
    arguments={"asset_id": "SE-LESTE-03", "description": "Substituição urgente", "priority": "high"},
)
print(json.dumps(r, indent=2)[:500])
assert r["decision"] == "ALLOW"
print("\n✓ Carlos pode criar com priority=high (P8 não aplica para managers)")

## 🎓 What you learned

- Cedar policies podem inspecionar `context.input.<param>`
- Permite controle granular alism de identity
- P8 is o tipo de policy que cria spans **AuthorizeAction com decision=DENY** (vs PartiallyAuthorizeActions que filtra por identity)

## Cleanup

```python
from utils import cleanup_policy_engine
cleanup_policy_engine(cfg["POLICY_STORE_ID"], region=region)
```

## Next

➡️ [Lab 04 — AgentCore Memory](../04-AgentCore-Memory/)